# CBSC Strategy Dashboard

This notebook demonstrates the interactive Dash dashboard for real-time strategy monitoring and performance analysis.

## Overview

The `StrategyDashboard` class provides:
- Interactive candlestick, line, and heatmap charts
- Real-time performance metrics display
- Dark/light theme switching
- WebSocket live data streaming
- Backtest results comparison
- Export functionality (PNG, HTML)
- Jupyter notebook integration

## 1. Import Required Libraries

Import the dashboard module and dependencies.

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Import dashboard components
from cbsc_strategy_sdk.dashboard import (
    StrategyDashboard,
    create_dashboard,
    ThemeManager,
    ChartBuilders,
    MetricsDisplay,
    LiveUpdateComponent,
)

print("✓ Dashboard module imported successfully")

## 2. Create Sample Data

Generate sample OHLCV price data and returns for demonstration.

In [ ]:
# Generate sample price data
np.random.seed(42)
dates = pd.date_range(start='2024-01-01', periods=252, freq='D')

# Simulate price movements
returns = np.random.randn(252) * 0.02
close_prices = 100 * (1 + returns).cumprod()

# Create OHLCV data
price_data = pd.DataFrame({
    'timestamp': dates,
    'open': close_prices * (1 + np.random.randn(252) * 0.005),
    'high': close_prices * (1 + abs(np.random.randn(252)) * 0.01),
    'low': close_prices * (1 - abs(np.random.randn(252)) * 0.01),
    'close': close_prices,
    'volume': np.random.randint(1000000, 10000000, 252),
})

# Create returns series
returns_series = pd.Series(returns, index=dates)

print(f"Generated {len(price_data)} days of data")
print(f"\nPrice Data Preview:")
print(price_data.head())
print(f"\nReturns Summary:")
print(f"Mean: {returns_series.mean():.4f}")
print(f"Std: {returns_series.std():.4f}")
print(f"Total Return: {(1 + returns_series).prod() - 1:.2%}")

## 3. Theme Management

Explore the theme manager for dark/light mode switching.

In [ ]:
# Create theme manager
theme = ThemeManager(initial_theme="dark")

# Get current colors
colors = theme.get_colors()
print(f"Current theme: {theme.current_theme}")
print(f"\nColor Palette:")
for name, color in colors.items():
    print(f"  {name}: {color}")

# Toggle theme
print(f"\nToggling theme...")
new_theme = theme.toggle_theme()
print(f"New theme: {new_theme}")

## 4. Chart Building

Create various chart types using ChartBuilders.

In [ ]:
# Initialize chart builder with dark theme
chart_builder = ChartBuilders(theme_manager=theme)

# Create candlestick chart
candlestick_fig = chart_builder.candlestick_chart(
    price_data.set_index('timestamp'),
    title="Price Chart - Candlestick",
    show_volume=True,
)

# Display chart
candlestick_fig.show()

## 5. Performance Metrics

Calculate and display strategy performance metrics.

In [ ]:
# Create metrics display
metrics = MetricsDisplay(returns=returns_series, theme_manager=theme)

# Get calculated metrics
print("Performance Metrics:")
print("=" * 40)
for key, value in metrics._metrics.items():
    if isinstance(value, float):
        print(f"{key.replace('_', ' ').title()}: {value:.2f}%")
    else:
        print(f"{key.replace('_', ' ').title()}: {value}")

## 6. Equity Curve and Drawdown Charts

Visualize strategy performance over time.

In [ ]:
# Create equity curve
equity_fig = chart_builder.equity_curve(
    returns=returns_series,
    title="Strategy Equity Curve",
)
equity_fig.show()

# Create drawdown chart
drawdown_fig = chart_builder.drawdown_chart(
    returns=returns_series,
    title="Drawdown Analysis",
)
drawdown_fig.show()

## 7. Returns Distribution

Analyze the distribution of strategy returns.

In [ ]:
# Create returns distribution histogram
dist_fig = chart_builder.returns_distribution(
    returns=returns_series,
    title="Returns Distribution",
)
dist_fig.show()

## 8. Strategy Comparison

Compare multiple strategies side by side.

In [ ]:
# Create comparison strategy returns
benchmark_returns = returns_series * 0.7
strategy_b_returns = returns_series * 1.2

# Compare strategies
comparison_results = {
    'Strategy A': returns_series,
    'Benchmark': benchmark_returns,
    'Strategy B': strategy_b_returns,
}

comparison_fig = chart_builder.comparison_chart(
    results=comparison_results,
    title="Strategy Comparison",
)
comparison_fig.show()

## 9. Create Full Dashboard

Create a complete dashboard with all components.

In [ ]:
# Create dashboard instance
dashboard = create_dashboard(
    title="My Strategy Dashboard",
    theme="dark",
    port=8050,
)

# Load data into dashboard
dashboard.load_backtest_results(
    returns=returns_series,
    benchmark_returns=benchmark_returns,
)

print("✓ Dashboard created successfully")
print(f"✓ Data loaded: {len(returns_series)} data points")

## 10. Run Dashboard in Jupyter

Start the dashboard server and display inline.

**Note:** This will start a Dash server. The dashboard will be displayed below.

In [ ]:
# Run dashboard in inline mode (for Jupyter)
# In a real notebook, uncomment the following line:
# dashboard.run(mode='inline')

# For now, let's just display the app instance info
print(f"Dashboard Title: {dashboard.title}")
print(f"Current Theme: {dashboard.current_theme}")
print(f"Port: {dashboard.port}")
print(f"\nTo start the dashboard, run:")
print(f"  dashboard.run(mode='inline')")
print(f"\nOr for external mode:")
print(f"  dashboard.run(mode='external')")

## 11. Live Updates with WebSocket

Setup real-time data streaming (optional).

In [ ]:
# Create live update component
live_component = LiveUpdateComponent(
    update_interval=1000,  # 1 second
    theme_manager=theme,
)

# Setup WebSocket (commented out - requires actual WebSocket server)
# live_component.setup_websocket(
#     ws_url="ws://localhost:8000/ws",
#     symbols=["AAPL", "MSFT"],
# )

# Create sample live data for demonstration
sample_live_data = live_component.create_sample_live_data(
    num_points=100,
    symbols=["AAPL"],
)

print(f"Generated sample live data: {len(sample_live_data)} points")
print(f"\nSample data preview:")
print(sample_live_data.head())

## 12. Export Dashboard

Export dashboard charts to PNG or HTML.

In [ ]:
# Export individual charts to HTML
# (Requires kaleido for PNG export)

print("Export Options:")
print("1. Export to HTML: dashboard.export_html('dashboard.html')")
print("2. Export to PNG: dashboard.export_png('dashboard.png')")
print("3. Export individual figures: fig.write_html('chart.html')")

# Example: Export equity curve to HTML
output_file = "equity_curve.html"
equity_fig.write_html(output_file)
print(f"\n✓ Exported equity curve to {output_file}")

## 13. Advanced Metrics Comparison

Compare strategy performance against a benchmark.

In [ ]:
# Create comparison chart with metrics table
comparison_fig = metrics.comparison_chart(
    benchmark_returns=benchmark_returns,
    title="Strategy vs Benchmark",
)
comparison_fig.show()

## 14. Monthly Returns Heatmap

Visualize monthly returns distribution.

In [ ]:
# Create monthly returns heatmap
heatmap_fig = metrics.monthly_returns_heatmap(
    title="Monthly Returns (%)",
)
heatmap_fig.show()

## 15. Rolling Metrics Chart

Display rolling Sharpe ratio and drawdown over time.

In [ ]:
# Create rolling metrics chart
rolling_fig = metrics.rolling_metrics_chart(
    window=60,  # 60 trading days
    title="Rolling Metrics (60-day window)",
)
rolling_fig.show()

## Summary

This notebook demonstrated:

1. **Theme Management** - Dark/light theme switching
2. **Chart Types** - Candlestick, line, heatmap charts
3. **Performance Metrics** - Comprehensive strategy analysis
4. **Live Updates** - Real-time WebSocket integration
5. **Dashboard Creation** - Full interactive dashboard
6. **Export Functionality** - PNG and HTML export

### Next Steps

- Connect to real CBSC backend API for live data
- Integrate with backtest results from Task #89
- Add custom chart types and indicators
- Deploy dashboard as standalone web application

### Requirements

Install required dependencies:
```bash
pip install dash dash-bootstrap-components plotly websockets pandas
```

For PNG export, also install:
```bash
pip install kaleido
```